# 03 事务

## 八股问答

### 1. ACID 分别由谁保证？

| 特性 | 含义 | 主要保证机制 |
| --- | --- | --- |
| 原子性 Atomicity | 事务要么全成功要么全回滚 | undo log 记录回滚版本 |
| 一致性 Consistency | 事务前后数据满足约束 | 应用层 + 数据库约束共同保证 |
| 隔离性 Isolation | 并发事务互不干扰 | 锁 + MVCC |
| 持久性 Durability | 提交后不丢失 | redo log（WAL） |

面试加分点：原子性和持久性靠日志，隔离性靠锁和 MVCC，一致性是前三者的结果，
不是某一个组件单独提供的。

### 2. 四种隔离级别和三种问题？

| 隔离级别 | 脏读 | 不可重复读 | 幻读 |
| --- | --- | --- | --- |
| 读未提交 READ UNCOMMITTED | 会 | 会 | 会 |
| 读已提交 READ COMMITTED | 不会 | 会 | 会 |
| 可重复读 REPEATABLE READ | 不会 | 不会 | 会（InnoDB 下已解决） |
| 串行化 SERIALIZABLE | 不会 | 不会 | 不会 |

三种问题的精确定义：

- 脏读：读到另一个事务未提交的数据，之后对方回滚，读到的就是假数据。
- 不可重复读：同一事务内两次读同一行，值不一样（另一个事务修改并提交了）。
- 幻读：同一事务内两次范围查询，行数不一样（另一个事务插入或删除了记录）。

### 3. MySQL 默认隔离级别是什么？为什么？

默认是 REPEATABLE READ。原因有历史因素（binlog 早期格式下 RC 会出现主从不一致），
更重要的是 InnoDB 用 MVCC + 间隙锁在 RR 下同时解决了幻读，所以 RR 和
SERIALIZABLE 之间没有明显的并发损失。

### 4. RR 怎么解决幻读？

两个层面：

- 快照读（普通 SELECT）：靠 MVCC，事务第一次读时生成 ReadView，之后一直读
  同一个快照，其他事务的插入看不见。
- 当前读（`SELECT ... FOR UPDATE`、`UPDATE`、`DELETE`）：靠间隙锁和临键锁，
  把查询范围内的间隙锁住，阻止其他事务插入。

所以面试要区分“普通 SELECT 查不到幻读”和“加锁读被间隙锁保护”两个答案。

### 5. Read View 是什么？

Read View 是事务做快照读时生成的“哪些事务还没提交”的视图，包含未提交事务
ID 列表、最小和最大事务 ID、当前事务 ID。普通 SELECT 沿 undo 版本链找第一个
对当前 Read View 可见的版本，从而不加锁也能读到一致快照。RC 每次读都生成，
RR 只在第一次读生成。详细结构和可见性规则见 [05-MVCC](05-MVCC.md)。

### 6. 可重复读下执行 UPDATE 会发生什么？

UPDATE 是当前读，不读快照，而是给目标行加 X 锁（RR 下为临键锁）后读最新已提交
版本，再写 undo、生成新版本。所以即使事务先做了快照读，UPDATE 仍基于最新值
执行；若目标范围被其他事务用间隙锁锁住，UPDATE 会阻塞等待。这也是 RR 下并发
更新不会互相覆盖的原因。

### 7. 隔离级别越高越好吗？

不是。隔离级别是并发安全性和吞吐量的取舍：级别越高锁越重、阻塞越多。业务里
多数场景 RC 或 RR 足够，串行化几乎只在极强一致要求下使用，例如库存扣减配合
行锁而不是把整个表串行化。


## 面试话术

“默认 RR 下，普通 SELECT 靠 MVCC 快照保证可重复读和防幻读；UPDATE、DELETE 和
加锁读走当前读，靠间隙锁和临键锁防幻读。如果业务能接受读到最新已提交数据，
用 RC 可以降低间隙锁范围，减少锁冲突。ACID 里原子性和持久性分别由 undo log
和 redo log 保证，隔离性由锁和 MVCC 保证，一致性是结果而不是单独组件。”


## 自查清单

- [ ] 能不看表说出四种隔离级别与三种问题的对应关系
- [ ] 能在双终端复现脏读、不可重复读、幻读
- [ ] 能解释 RR 下普通 SELECT 为什么两次结果一致
- [ ] 能说清快照读和当前读在防幻读上的分工
